# 01 · Prepare — Is the Google Merchandise Store data fit for the decision?

**Business task** ([reports/01_ask.md](../reports/01_ask.md)): help the Head of Marketing decide (D1) how to reallocate channel budget and (D2) which first-time visitors are worth remarketing to.

This notebook profiles `bigquery-public-data.google_analytics_sample.ga_sessions_*` for credibility and completeness. Every query lives in [`sql/prepare/`](../sql/prepare/). Results are cached to `data/raw/`, so rerunning costs nothing. Delete a cache file or pass `refresh=True` to re-query BigQuery.

Full write-up: [reports/02_prepare.md](../reports/02_prepare.md).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import bq
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Referral traffic whose source is hidden

38% of all revenue arrives as `Referral` with source `(direct)`, which is a contradiction. Breaking it down by referral path and by share of sessions from Google office cities:

In [2]:
bq.query("prepare/p01_referral_direct_paths.sql", cache="p01_referral_direct_paths")

,referral_path,sessions,visitors,conv_pct,pct_google_office_cities,revenue_usd
0,/,67236,37332,7.61,65.8,676328.0
1,/offer/2145,1064,720,1.69,63.2,2300.0
2,/Rewards/Products/Details.aspx,185,166,2.16,29.7,219.0
3,/pagead/ads,84,50,0.00,96.4,NaN
4,/da/b/dabAdPreview,47,19,0.00,55.3,NaN
5,/Client.html,45,36,0.00,6.7,NaN
6,/safeframe/1-0-5/html/container.html,34,24,0.00,55.9,NaN
7,/cm/CampaignMgmt,24,14,0.00,58.3,NaN
8,/pagead/gadgets/dynamic_in_page_V1/Responsive_...,22,17,0.00,59.1,NaN
9,/pagead/gadgets/dynamic_in_page_V1/Responsive_...,20,11,0.00,65.0,NaN


Path `/` accounts for 97.5% of these sessions. It converts at 7.6% (10× the external rate), and 66% of its sessions come from Google office cities. This points to Google's internal employee store link. Notebook 03 confirms it: in the lab's unredacted table, these sessions come from `mall.googleplex.com`.

**Decision D-P1:** flag these visitors as *likely internal* at the **visitor** level, so their later bookmark visits don't inflate Direct.

In [3]:
bq.query("prepare/p02_internal_segment_size.sql", cache="p02_internal_segment_size")

,segment,visitors,sessions,purchase_sessions,conv_pct,revenue_usd,non_entry_direct_sessions
0,Internal (likely employees),37332,76536,5408,7.07,730377.0,3023
1,External,676835,827117,6144,0.74,1049773.0,140003


About 5% of visitors produce **47% of purchase sessions and 41% of revenue**. Leaving them in would make 'Referral' look like the best channel, and marketing can't buy those visits.

## 2. Tracking continuity

In [4]:
bq.query("prepare/p03_monthly_trend.sql", cache="p03_monthly_trend")

,month,sessions,purchase_sessions,conv_pct,revenue_usd,pct_sessions_with_product_view
0,2016-08,74759,1145,1.53,189977.0,19.1
1,2016-09,71032,870,1.22,136614.0,17.4
2,2016-10,97506,872,0.89,124416.0,10.9
3,2016-11,113972,919,0.81,132891.0,8.9
4,2016-12,79124,1395,1.76,167324.0,14.2
5,2017-01,64694,697,1.08,106248.0,14.3
6,2017-02,62192,708,1.14,116112.0,13.4
7,2017-03,69931,883,1.26,150225.0,12.8
8,2017-04,67126,928,1.38,222768.0,13.8
9,2017-05,65371,1115,1.71,135952.0,15.1


There are no gaps, and Enhanced Ecommerce product tracking appears in every month. 2017-08 is a single day. Oct–Nov 2016 shows a low-intent traffic spike, which Analyze investigates.

## 3. Field completeness

In [5]:
bq.query("prepare/p04_field_completeness.sql", cache="p04_field_completeness").T

,0
sessions,903653
pct_null_pageviews,0.01
pct_null_time_on_site,49.99
pct_bounces,49.87
pct_country_not_set,0.16
pct_city_redacted,56.24
pct_browser_redacted,0.0
pct_keyword_not_provided,24.63
pct_campaign_not_set,96.9
pct_purchase_without_revenue,0.0041


- `timeOnSite` is NULL on 49.99% of sessions, almost exactly the bounce rate, so NULL means a single-hit session (**impute 0**).
- City is 56% redacted and campaign is 97% not set, so we **work at channel and country level**.
- The 898 duplicate keys are checked below.

In [6]:
bq.query("prepare/p07_duplicate_sessions.sql", cache="p07_duplicate_sessions")

,n_rows,n_start_times,day_span,keys,transactions
0,2,2,1,898,41


All 898 appear twice, with **different start times one day apart**: these are sessions that crossed midnight, which GA splits. Unique key = `(fullVisitorId, visitId, visitStartTime)`.

## 4. Very large orders

In [7]:
bq.query("prepare/p05_order_values.sql", cache="p05_order_values")

,channel,purchase_sessions,median_order_usd,p99_order_usd,max_order_usd,pct_revenue_from_orders_over_5k
0,Referral,5322,62.0,1182.0,8258.0,4.7
1,Organic Search,3443,44.0,1015.0,9228.0,5.8
2,Direct,2061,66.0,1961.0,25249.0,15.8
3,Paid Search,469,45.0,798.0,1526.0,0.0
4,Display,143,85.0,32154.0,47082.0,72.7
5,Social,104,39.0,902.0,1224.0,0.0
6,Affiliates,9,30.0,243.0,243.0,0.0
7,(Other),1,12.0,12.0,12.0,0.0


73% of Display's revenue comes from orders over \$5k. **Decision D-P2:** attribution counts conversions first, with revenue capped at the 99th percentile shown second.

## 5. Time to purchase → window length

In [8]:
bq.query("prepare/p06_days_to_purchase.sql", cache="p06_days_to_purchase").T

,0
buyers,5047
pct_same_visit,57.3
pct_within_1d,16.8
cum_pct_7d,85.6
cum_pct_14d,90.2
cum_pct_30d,95.1
cum_pct_60d,97.8
cum_pct_90d,99.0


In [9]:
bq.query("prepare/p08_external_first_vs_later.sql", cache="p08_external_first_vs_later").T

,0
external_new_visitors,668583
external_new_who_returned,74349
bought_first_visit,2893
bought_later_visit,2154
bought_later_within_30d,1908


A **30-day window captures 95%** of external purchases and matches GA's default lookback. About 1,900 external first-time visitors buy on a later visit within 30 days, a base rate near 0.3%, so the D2 model is judged by PR-AUC and lift, not accuracy.

## 6. Audit of Google's lab label (GSP229)

Google's lab *Predict Visitor Purchases with a Classification Model in BigQuery ML* uses the label `will_buy_on_return_visit` on first visits. Rebuilding its label and splits, then adding our internal flag:

In [10]:
bq.query("audit/a01_lab_label_audit.sql", cache="a01_lab_label_audit")

,split,n_rows,positives,positive_rate_pct,internal_positives,pct_positives_internal,external_positive_rate_pct
0,train,550037,3801,0.691,2025,53.3,0.338
1,eval,96470,852,0.883,481,56.5,0.410
2,predict,56553,317,0.561,164,51.7,0.284


On the public sample, **53% of the positives under the lab's label are likely Google employees**.

This is an estimate: the lab trains on a different table (`data-to-insights.ecommerce.web_analytics`), which shows the referrers that the public sample hides. [Notebook 03](03_analyze_lab_audit.ipynb) repeats the audit on the lab's own table with verified employee flags (61% of training positives) and tests how much of the lab model's 0.91 ROC-AUC depends on them (H4).

In [11]:
bq.query("audit/a02_lab_label_by_month.sql", cache="a02_lab_label_by_month")

,first_visit_month,external_first_visits,lab_label_rate_pct
0,2016-08,55051,0.392
1,2016-09,52810,0.390
2,2016-10,77635,0.318
3,2016-11,91782,0.228
4,2016-12,54399,0.426
5,2017-01,46293,0.300
6,2017-02,45789,0.397
7,2017-03,51936,0.329
8,2017-04,49785,0.350
9,2017-05,45271,0.453


Because the lab's label has no fixed window, later cohorts have less time to return. The effect is visible only in the last month (2017-07/08), so it's a minor issue compared with the internal traffic.